# Ajuste MAS — sistema biela-manivela
**Óptica y Ondas / UTEC 2026-II**

Cubre los puntos **1 (movimiento angular y lineal)** y **2 (espacio angular vs lineal)** de la guía.

> ⚠️ **Diferencia con la plantilla oficial.** Nuestros `.txt` salieron con el orden
> `Tiempo, Posición, Velocidad1, Aceleración1, Ángulo, Velocidad2, Aceleración2`.
> La plantilla del profesor asume `Tiempo, Ángulo, ..., Posición`. Aquí las columnas ya
> están mapeadas correctamente: `a` = aceleración **lineal** (m/s²), `alpha` = aceleración
> **angular** (rad/s²).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import curve_fit

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Orden REAL de columnas en nuestros archivos de Logger Pro
COLUMNAS = ["t", "x", "v", "a", "theta", "omega", "alpha"]
#            s   m  m/s  m/s²   rad     rad/s    rad/s²

CORRIDAS = {
    "10Hz-c1": "biela-manivela-10Hz-08-09-2026-c1.txt",
    "10Hz-c2": "biela-manivela-10Hz-08-09-2026-c2.txt",
    "20Hz-c1": "biela-manivela-20Hz-08-09-2026-c1.txt",
    "20Hz-c2": "biela-manivela-20Hz-08-09-2026-c2.txt",
    "30Hz-c1": "biela-manivela-30Hz-08-09-2026-c1.txt",   # DESCARTADA: ver data/NOTAS.md
    "30Hz-c2": "biela-manivela-30Hz-08-09-2026-c2.txt",
}

def dir_datos():
    # Encuentra data/raw sin importar desde donde se ejecute el notebook
    for c in [Path("../data/raw"), Path("data/raw"), Path(".")]:
        if (c / CORRIDAS["10Hz-c2"]).exists():
            return c
    raise FileNotFoundError("No encuentro data/raw. Ejecuta el notebook desde "
                            "notebooks/ o desde la raiz del repo.")

DIR_FIG = Path("../figuras") if Path("../figuras").exists() else Path("figuras")
DIR_FIG.mkdir(exist_ok=True)

def guardar(nombre):
    # Guarda la figura actual en figuras/ para usarla en el documento
    p = DIR_FIG / f"{nombre}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"figura guardada: {p}")

def cargar(corrida):
    """Lee un .txt de Logger Pro y devuelve un dict con las 7 columnas."""
    d = np.genfromtxt(dir_datos() / CORRIDAS[corrida],
                      skip_header=7, encoding="utf-8-sig")
    d = d[~np.isnan(d).any(axis=1)]          # descarta filas incompletas
    return {c: d[:, i] for i, c in enumerate(COLUMNAS)}

def R2(y, y_fit):
    return 1 - np.sum((y - y_fit)**2) / np.sum((y - y.mean())**2)

## 1. Elegir archivo e intervalo

Los intervalos sugeridos evitan los transitorios de arranque y frenado.
`30Hz-c1` está excluida: los primeros 0.7 s el carrito está quieto y la señal
tiene ruido que domina el espectro. Ver `data/NOTAS.md`.

In [ ]:
CORRIDA = "10Hz-c2"       # 10Hz-c1 | 10Hz-c2 | 20Hz-c1 | 20Hz-c2 | 30Hz-c2

# Ventanas donde el giro fue mas parejo (~2.5 vueltas, R2 > 0.985 en el ajuste MAS).
# Para la FFT conviene una ventana mas larga: ver el otro notebook.
INTERVALOS = {"10Hz-c1": (5.0, 11.0),
              "10Hz-c2": (2.0,  9.0),
              "20Hz-c1": (5.5, 12.5),
              "20Hz-c2": (4.5, 11.5),
              "30Hz-c2": (1.0,  9.0)}
if CORRIDA not in INTERVALOS:
    raise ValueError(f"{CORRIDA} no es analizable (ver data/NOTAS.md). "
                     f"Usa una de: {list(INTERVALOS)}")
T_MIN, T_MAX = INTERVALOS[CORRIDA]

L_BIELA = 0.36            # m, medido con regla

D = cargar(CORRIDA)
m = (D["t"] >= T_MIN) & (D["t"] <= T_MAX)
t, x, v, a       = D["t"][m], D["x"][m], D["v"][m], D["a"][m]
th, om, al       = D["theta"][m], D["omega"][m], D["alpha"][m]

dt = np.median(np.diff(t)); fs = 1/dt
vueltas = abs(th[-1] - th[0]) / (2*np.pi)
print(f"{CORRIDA}  |  {len(t)} muestras  |  fs = {fs:.1f} Hz  |  "
      f"t = [{t[0]:.2f}, {t[-1]:.2f}] s  |  {vueltas:.2f} vueltas")

## 2. Parámetro geométrico r a partir de los datos

El recorrido total del carrito es exactamente **2r**, así que podemos obtener el radio
de la manivela sin depender de la medición con regla.

In [ ]:
recorrido = x.max() - x.min()
r_exp = recorrido / 2

print(f"Recorrido del carrito : {recorrido*100:.2f} cm")
print(f"r experimental        : {r_exp*100:.2f} cm   (medido con regla: 12.5 cm)")
print(f"L de la biela         : {L_BIELA*100:.1f} cm")
print(f"Razón r/L             : {r_exp/L_BIELA:.3f}   "
      f"{'OK, < 1/3' if r_exp/L_BIELA < 1/3 else 'ATENCION: > 1/3'}")

## 3. Punto 1a — Movimiento angular: ¿fue un MCU?

Si el giro fuera MCU perfecto, θ(t) sería una recta, ω constante y α ≡ 0.
Medimos cuánto nos alejamos de eso.

In [ ]:
pend = np.polyfit(t, th, 1)          # ajuste lineal de theta(t)
om_lin = pend[0]
om_m, om_s = np.abs(om).mean(), np.abs(om).std()
desv = 100 * om_s / om_m

fig, ax = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)

ax[0].plot(t, th, ".-", ms=3, lw=.8, label="datos")
ax[0].plot(t, np.polyval(pend, t), "r--", lw=1.8,
           label=f"MCU ideal: ω = {om_lin:.3f} rad/s")
ax[0].set_ylabel("θ (rad)"); ax[0].legend()
ax[0].set_title(f"Movimiento angular — {CORRIDA}")

ax[1].plot(t, om, ".-", ms=3, lw=.8, color="tab:green", label="datos")
ax[1].axhline(om.mean(), ls="--", c="r", lw=1.8,
              label=f"media = {om.mean():.3f} rad/s")
ax[1].fill_between(t, om.mean()-om_s, om.mean()+om_s, alpha=.15, color="r",
                   label=f"±1σ ({desv:.1f} %)")
ax[1].set_ylabel("ω (rad/s)"); ax[1].legend()

ax[2].plot(t, al, ".-", ms=3, lw=.8, color="tab:red")
ax[2].axhline(0, ls="--", c="k", lw=1.2, label="α = 0 (MCU ideal)")
ax[2].set_ylabel("α (rad/s²)"); ax[2].set_xlabel("Tiempo (s)"); ax[2].legend()

plt.tight_layout(); guardar(f"01_angular_{CORRIDA}"); plt.show()

print(f"ω medio          : {om_m:.3f} rad/s")
print(f"Desv. estándar   : {om_s:.3f} rad/s  ({desv:.1f} % del valor medio)")
print(f"Frecuencia f1    : {om_m/(2*np.pi):.3f} Hz   (periodo {2*np.pi/om_m:.2f} s)")
print(f"Fracción de tiempo con α > 0 : {(al>0).mean():.2f}  (MCU ideal: 0.50)")

**Para comentar en el video (punto 1, intervalos de α positiva y negativa):**

- α alterna de signo de forma cuasi-periódica: no es ruido, es el ritmo de la mano
  acelerando y frenando dentro de cada vuelta.
- La fracción de tiempo con α>0 cercana a 0.5 indica que las aceleraciones y
  desaceleraciones se compensan: **en promedio** ω se mantiene.
- Conclusión honesta: el movimiento es *aproximadamente* MCU, con una desviación del
  orden del valor impreso arriba. No es MCU estricto, y eso tiene consecuencia directa
  en el espectro (los picos se ensanchan).

## 4. Punto 1b — Ajuste no lineal con las ecuaciones del MAS

In [ ]:
def x_mas(t, A0, A, w, phi): return A0 + A*np.cos(w*t + phi)
def v_mas(t, A, w, phi):     return -w*A*np.sin(w*t + phi)
def a_mas(t, A, w, phi):     return -w**2*A*np.cos(w*t + phi)

# Condiciones iniciales razonables a partir de los propios datos
p0 = [(x.max()+x.min())/2, (x.max()-x.min())/2, om_m, 0.0]

popt_x, pcov_x = curve_fit(x_mas, t, x, p0=p0)
A0_f, A_f, w_f, phi_f = popt_x
err_x = np.sqrt(np.diag(pcov_x))

popt_v, pcov_v = curve_fit(v_mas, t, v, p0=[A_f, w_f, phi_f])
popt_a, pcov_a = curve_fit(a_mas, t, a, p0=[A_f, w_f, phi_f])

print("AJUSTE MAS")
print(f"  x(t):  A0 = {A0_f:.4f} ± {err_x[0]:.4f} m")
print(f"         A  = {A_f:.4f} ± {err_x[1]:.4f} m   (esperado ≈ r = {r_exp:.4f} m)")
print(f"         ω  = {w_f:.4f} ± {err_x[2]:.4f} rad/s")
print(f"         φ  = {phi_f:.4f} rad")
print(f"  v(t):  A = {popt_v[0]:.4f}, ω = {popt_v[1]:.4f}, φ = {popt_v[2]:.4f}")
print(f"  a(t):  A = {popt_a[0]:.4f}, ω = {popt_a[1]:.4f}, φ = {popt_a[2]:.4f}")
print()
print(f"  R² posición    : {R2(x, x_mas(t, *popt_x)):.4f}")
print(f"  R² velocidad   : {R2(v, v_mas(t, *popt_v)):.4f}")
print(f"  R² aceleración : {R2(a, a_mas(t, *popt_a)):.4f}   <- el peor: ahí vive el 2° armónico")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

for axi, (dat, fit, lab, u) in zip(ax, [
        (x, x_mas(t, *popt_x), "Posición",    "m"),
        (v, v_mas(t, *popt_v), "Velocidad",   "m/s"),
        (a, a_mas(t, *popt_a), "Aceleración", "m/s²")]):
    axi.plot(t, dat, "o", ms=3, alpha=.55, label="datos")
    axi.plot(t, fit, "-", lw=2, color="crimson", label=f"MAS (R² = {R2(dat, fit):.4f})")
    axi.set_ylabel(f"{lab} ({u})"); axi.legend(loc="upper right")

ax[0].set_title(f"Ajuste MAS de x(t), v(t), a(t) — {CORRIDA}")
ax[2].set_xlabel("Tiempo (s)")
plt.tight_layout(); guardar(f"02_ajuste_MAS_{CORRIDA}"); plt.show()

## 5. Punto 2 — Espacio angular vs lineal: x(θ)

Aquí es donde la anarmonicidad se ve mejor, porque al graficar contra θ
eliminamos el efecto de que ω no sea constante.

In [ ]:
def x_theta_mas(th, A0, A, phi):
    return A0 + A*np.cos(th + phi)

popt_t, _ = curve_fit(x_theta_mas, th, x, p0=[x.mean(), r_exp, 0.0])
fit_mas = x_theta_mas(th, *popt_t)
res_mas = x - fit_mas

orden = np.argsort(th % (2*np.pi))
thw = (th % (2*np.pi))[orden]

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1.4]})
ax[0].plot(thw, x[orden], "o", ms=3.5, alpha=.5, label="datos x(θ)")
ax[0].plot(thw, fit_mas[orden], "-", lw=2, color="crimson",
           label=f"A₀ + A·cos(θ+φ)   R² = {R2(x, fit_mas):.4f}")
ax[0].set_ylabel("Posición (m)"); ax[0].legend()
ax[0].set_title(f"Posición del pistón vs ángulo de la manivela — {CORRIDA}")

ax[1].plot(thw, 1000*res_mas[orden], "o", ms=3.5, alpha=.6, color="tab:purple")
ax[1].axhline(0, c="k", lw=.8)
ax[1].set_ylabel("Residuo (mm)"); ax[1].set_xlabel("θ mod 2π (rad)")
ax[1].set_xticks(np.arange(0, 2.1*np.pi, np.pi/2))
ax[1].set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
plt.tight_layout(); guardar(f"03_x_vs_theta_{CORRIDA}"); plt.show()

print(f"A del ajuste : {popt_t[1]:.4f} m   (|A| = {abs(popt_t[1])*100:.2f} cm)")
print(f"RMS residuo  : {1000*res_mas.std():.2f} mm")
print("El residuo NO es ruido aleatorio: oscila con el DOBLE de frecuencia que θ.")
print("Eso es exactamente el segundo armónico.")

## 6. Punto 2 — Modelo corregido con el segundo armónico

Añadimos el término anarmónico y comparamos.

> **Nota sobre el signo.** El desarrollo exacto de x = r·cosθ + √(L² − r²sin²θ) da
> x ≈ (L − r²/4L) + r·cosθ **+** (r²/4L)·cos2θ. La guía lo escribe con signo negativo
> (ec. 18 y 20) porque en la ec. 15 aparece `cos²θ` donde debería ir `sin²θ`.
> Además, nuestro sensor mide **distancia al carrito**, así que el eje está invertido
> respecto al x geométrico de la figura 1 y el signo ajustado se invierte otra vez.
> Lo que hay que comparar con la teoría es la **magnitud** |B| vs r²/(4L).

In [ ]:
def x_theta_2arm(th, A0, A, phi, B):
    return A0 + A*np.cos(th + phi) + B*np.cos(2*(th + phi))

popt_2, _ = curve_fit(x_theta_2arm, th, x, p0=[*popt_t, 0.005])
fit_2 = x_theta_2arm(th, *popt_2)
res_2 = x - fit_2

A_aj, B_aj = abs(popt_2[1]), abs(popt_2[3])
B_teo = A_aj**2 / (4*L_BIELA)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1.4]})
ax[0].plot(thw, x[orden], "o", ms=3.5, alpha=.45, label="datos")
ax[0].plot(thw, fit_mas[orden], "--", lw=1.8, color="crimson",
           label=f"MAS puro (R² = {R2(x, fit_mas):.4f})")
ax[0].plot(thw, fit_2[orden], "-", lw=2.2, color="tab:blue",
           label=f"MAS + 2° armónico (R² = {R2(x, fit_2):.4f})")
ax[0].set_ylabel("Posición (m)"); ax[0].legend()
ax[0].set_title("Efecto de incluir el término anarmónico")

ax[1].plot(thw, 1000*res_mas[orden], "o", ms=3, alpha=.45, color="crimson",
           label=f"residuo MAS ({1000*res_mas.std():.2f} mm rms)")
ax[1].plot(thw, 1000*res_2[orden], "o", ms=3, alpha=.6, color="tab:blue",
           label=f"residuo con 2° arm. ({1000*res_2.std():.2f} mm rms)")
ax[1].axhline(0, c="k", lw=.8); ax[1].legend(fontsize=8)
ax[1].set_ylabel("Residuo (mm)"); ax[1].set_xlabel("θ mod 2π (rad)")
ax[1].set_xticks(np.arange(0, 2.1*np.pi, np.pi/2))
ax[1].set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
plt.tight_layout(); guardar(f"04_segundo_armonico_{CORRIDA}"); plt.show()

print(f"Amplitud 1er armónico  |A| = {A_aj*100:.2f} cm")
print(f"Amplitud 2do armónico  |B| = {B_aj*100:.3f} cm")
print(f"Valor teórico r²/(4L)      = {B_teo*100:.3f} cm")
print(f"Discrepancia               = {100*abs(B_aj-B_teo)/B_teo:.1f} %")
print()
print(f"Reducción del residuo: {1000*res_mas.std():.2f} mm  ->  {1000*res_2.std():.2f} mm")

## 7. Qué decir en el video sobre estos dos bloques

**Punto 1 — movimiento angular y lineal**
- θ(t) es casi una recta → giro aproximadamente uniforme; cuantificar con la desviación de ω.
- α alterna de signo dentro de cada vuelta (la mano acelera y frena); en promedio se compensa.
- El MAS ajusta muy bien la posición, algo peor la velocidad y claramente peor la
  **aceleración**. Ese deterioro progresivo no es casual: al derivar dos veces, el término
  cos2θ se multiplica por 4, así que la anarmonicidad se amplifica.

**Punto 2 — espacio angular vs lineal**
- x(θ) *parece* sinusoidal a simple vista, pero el residuo delata la asimetría.
- El residuo del ajuste MAS oscila dos veces por vuelta → es una señal, no ruido.
- Al agregar el término cos2θ el R² sube y el residuo cae varias veces.
- La asimetría física: con la biela de longitud finita, el pistón tarda distinto en ir de
  un extremo al otro. Si L → ∞ el término desaparece y recuperamos el MAS puro.

**Efecto de la frecuencia de muestreo (comparar las tres corridas)**
- El R² de la aceleración empeora notablemente al pasar de 10 Hz a 20 y 30 Hz.
- No es que el experimento saliera peor: Logger Pro obtiene la aceleración
  **derivando numéricamente dos veces** la posición, y al reducir Δt el ruido de
  cuantización del sensor se amplifica como 1/Δt². Por eso a mayor fs la señal de
  aceleración está más sucia.
- Es un buen punto de discusión: más muestras no siempre significa mejor señal.